# Pychometrics Tutorial

This notebook walks through the full llm tracker workflow:
1. **LLM Encoding** — run a language model over a directory of interviews to identify psychological construct instances
    * Select a model from https://openrouter.ai/models, and have an openrouter API key
2. **Human Encoding** — produce a second set of encodings for comparison.
    * For the purpsoes of the tutorial we will use a second LLM as the dummy human, but feel free to use any human data you have.
3. **Comparison** — align the two sets of encodings instance-by-instance using an LLM matcher
4. **Summary Tables** — compute per-interview, concatenated, and weighted performance metrics

## Setup

In [9]:
import pandas as pd

from pychometrics import PychometricsAnalyzer
from pychometrics.comparison import (
    PychometricsComparator,
    format_comparison_table,
    compute_summary_tables,
    format_weighted_summary,
)

# Paths
* Define all relevant paths here, as well as your api key and the model(s) you want to use for encoding.
* input_dir:     directory of interview files (.txt or .csv), one file per interview
* codebook_path: JSON codebook defining the psychological constructs to identify
* human_dir:     directory of human-coded JSON encodings for comparison
                (one JSON per interview, same naming as input files)

In [ ]:

input_dir     = "/Users/freymon.perez/Downloads/sample_interviews"
codebook_path = "psychedelic_codebook.json"
human_dir     = "path/to/human_coded_encodings" # for comparison 

# API Key and Model selection.
# Uses OpenRouter. Any model available on OpenRouter can be passed to model_name.
# At the descretion of the user different models can be used for encoding or comparison, 
# In this tutorial we'll stick to one model for all tasks.
api_key   = "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"
llm_model = "google/gemini-3-flash-preview"

## 1. LLM Encoding

Runs the LLM over every file in `input_dir` and saves results to a timestamped output directory.

Output structure:
```
my_analysis_1_YYYY-MM-DD_HHMMSS/
    encodings/   <- one JSON per interview
    metadata/    <- token usage, latency, model info
    errors/      <- any failed documents
    README.md
```

In [4]:
analyzer = PychometricsAnalyzer(
    api_key=api_key,
    model_name=llm_model,
)

results, metadata, errors = analyzer.analyze_directory(
    input_dir=input_dir,
    codebook_path=codebook_path,
    output_dir="my_analysis_1",  # timestamped suffix added automatically
)

Processing [1/3]: s29_integration1.txt
  ✓ Found 25 construct instances
  Progress: Successful 1/3; Errors 0/3
Processing [2/3]: s29_integration2.txt
  ✓ Found 41 construct instances
  Progress: Successful 2/3; Errors 0/3
Processing [3/3]: s7_integration1.txt
  ✓ Found 26 construct instances
  Progress: Successful 3/3; Errors 0/3

Analysis complete!
  Output directory: /Users/freymon.perez/Projects/GitHub/pychometrics/my_analysis_1_2026-03-26_103705
  Successful: 3/3


### Retry Failed Documents

If any documents failed (API errors, malformed responses), retry them without re-processing successful ones.
Remove #s from the code below to re-run any errors if necessary. 

In [ ]:

# recovered_results, recovered_meta, remaining_errors = analyzer.retry_errors(
#     output_dir="path/to/my_analysis_1_YYYY-MM-DD_HHMMSS",
#     codebook_path=codebook_path,
# )

## 2. Human Encoding

For comparison you need a second set of encodings. In production this would be a directory of
human-coded JSONs. For testing, a second LLM run can serve as a stand-in.

The human encoding directory must follow the same file naming as `input_dir`.

**Note:** This block is only necessary to run in the case of no avaiable human data for testing.

In [5]:
# Example: use a second LLM run as a dummy human coder
human_analyzer = PychometricsAnalyzer(
    api_key=api_key,
    model_name=llm_model,
)

results_human, metadata_human, errors_human = human_analyzer.analyze_directory(
    input_dir=input_dir,
    codebook_path=codebook_path,
    output_dir="human_encodings",
)

Processing [1/3]: s29_integration1.txt
  ✓ Found 26 construct instances
  Progress: Successful 1/3; Errors 0/3
Processing [2/3]: s29_integration2.txt
  ✓ Found 62 construct instances
  Progress: Successful 2/3; Errors 0/3
Processing [3/3]: s7_integration1.txt
  ✓ Found 33 construct instances
  Progress: Successful 3/3; Errors 0/3

Analysis complete!
  Output directory: /Users/freymon.perez/Projects/GitHub/pychometrics/human_encodings_2026-03-26_104045
  Successful: 3/3


## 3. Comparison

Aligns human and LLM encodings across all interviews. For each construct, an LLM matcher
decides which human quotes and LLM quotes refer to the same passage, flagging paraphrases
and computing span overlap.

Each instance is classified as:
- **matched** — both coders found it (TP)
- **human_only** — human found it, LLM missed it (FN)
- **llm_only** — LLM found it, human missed it (FP)

In [ ]:
# Update these paths to point to the encodings/ subdirectories of your runs
# If this notebook is run multiple times, remember to update these paths.
llm_encoding_dir   = "/Users/freymon.perez/Projects/GitHub/pychometrics/my_analysis_1_2026-03-26_103705/encodings"
human_encoding_dir = "/Users/freymon.perez/Projects/GitHub/pychometrics/human_encodings_2026-03-26_104045/encodings"

comparator = PychometricsComparator(
    api_key=api_key,
    match_model=llm_model,
)

comparison_results = comparator.compare_directories(
    human_dir=human_encoding_dir,
    llm_dir=llm_encoding_dir,
    output_dir="comparison_run_new",  # optional; saves JSON comparison files
)

## 4. Summary Tables

Build a combined row-level DataFrame across all interviews, then compute the three summary tables:

| Table | Grain | Description |
|---|---|---|
| `per_interview` | (doc, construct) | Raw counts and metrics per interview |
| `concatenated` | construct | Counts pooled across all interviews |
| `weighted_summary` | construct | Weighted median [min–max] across interviews, weighted by union |

In [8]:
# Combine all interview comparison results into one DataFrame
df = pd.concat(
    [format_comparison_table(result) for result in comparison_results.values()],
    ignore_index=True,
)

per_interview, concatenated, weighted_summary = compute_summary_tables(df)

ValueError: No objects to concatenate

In [ ]:
# Row-level view: every matched/human_only/llm_only instance across all interviews
df

In [ ]:
# Per-interview metrics: one row per (interview, construct)
per_interview

In [ ]:
# Concatenated metrics: counts pooled across all interviews, one row per construct
concatenated

In [ ]:
# Weighted summary: median [min-max] weighted by union size, with n_docs prevalence column
format_weighted_summary(weighted_summary)